# Lecture: Conditional Diffusion & Classifier-Free Guidance

So far our diffusion models have been **unconditional**: we start from noise and
get *some* Fashion-MNIST item, with no control over *which* one. Real generative
systems — above all **Stable Diffusion** — let us *steer* generation: "a photo of
a cat", "a red sneaker". This notebook builds the two ingredients that make this
possible, on Fashion-MNIST:

1. **Conditioning.** We tell the network *which class* to generate by feeding a
   learned **class embedding** alongside the timestep. The architecture change is
   tiny: one `nn.Embedding`, added to the existing timestep embedding.

2. **Classifier-Free Guidance (CFG)** (Ho & Salimans, 2022). A clever trick that
   lets a *single* network act both **conditionally** and **unconditionally**,
   and then **amplifies** the difference between the two to make samples match
   the requested class more strongly. This is the `guidance_scale` knob you may
   know from Stable Diffusion.

Both ideas carry over **unchanged** to text-conditioned models — the only
difference there is that the class embedding is replaced by a text encoder. Once
you understand CFG on 10 clothing classes, you understand the core of prompt
guidance.

> **Note:** Unlike C4-3, this notebook needs a **new training run**, because the
> C4-2 model never saw any labels. Training a conditional model on a Colab T4 GPU
> takes roughly 12–18 minutes, or load the provided checkpoint.

Run the following cell only if you are working with Google Colab to copy the required .py file into the root directory. If you are working locally, ignore this cell.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/C4-Diffusion_Models/Diffusion.py ./

### Data Preparation

Same Fashion-MNIST setup as C4-2, but this time we **keep the labels** — they are
the conditioning signal. The ten classes are listed below; we will later ask the
model to generate a specific one.

In [ ]:
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import FashionMNIST
from Diffusion import DDPM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

TIMESTEPS   = 1000
CHANNELS    = 64
NUM_CLASSES = 10
BATCH_SIZE  = 128
EPOCHS      = 40
LR          = 2e-4

FASHION_CLASSES = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal",  "Shirt",   "Sneaker",  "Bag",   "Ankle boot"
]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_dataset = FashionMNIST(root="./data", train=True, download=True, transform=transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=0, pin_memory=True)

print(f"Training samples: {len(train_dataset)} | classes: {NUM_CLASSES}")

### Conditional Model

We build the same `DDPM` as before, but pass `num_classes=10`. Internally this
adds an `nn.Embedding(num_classes + 1, time_dim)` to the U-Net: each class gets a
learned vector that is **added to the timestep embedding**, so every residual
block is informed about the target class.

The "+1" is important: the extra index is a **null class** that means "no
condition". The model learns it via label dropout during training, and CFG uses
it at sampling time to obtain the unconditional prediction — all from one
network.

In [ ]:
model = DDPM(timesteps=TIMESTEPS, channels=CHANNELS, num_classes=NUM_CLASSES).to(device)

# Sanity check: the network now accepts labels y
_x = torch.randn(4, 1, 28, 28, device=device)
_t = torch.randint(0, TIMESTEPS, (4,), device=device)
_y = torch.randint(0, NUM_CLASSES, (4,), device=device)
print("Conditional prediction shape:", model.model(_x, _t, _y).shape)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

### Training with Label Dropout

The training objective is the same MSE noise-prediction loss as C4-2, with one
addition implemented in `DDPM.loss`: with probability `p_uncond` (default 0.1)
each label is replaced by the **null class**. As a result the *same* network
learns:

- the **conditional** score $\varepsilon_\theta(x_t, t, y)$ (90% of the time), and
- the **unconditional** score $\varepsilon_\theta(x_t, t, \varnothing)$ (10%).

Having both inside one model is exactly what classifier-free guidance needs — no
separate classifier required (hence the name).

> On a Colab T4, 40 epochs take ~12–18 minutes. Reduce `EPOCHS` if you are short
> on time, or skip to the pre-trained checkpoint below.

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=LR)

history = []
for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for x0, y in train_loader:
        x0 = x0.to(device, non_blocking=True)
        y  = y.to(device, non_blocking=True)

        loss = model.loss(x0, y, p_uncond=0.1)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()

    history.append(total / len(train_loader))
    print(f"Epoch {epoch+1:3d}  loss={history[-1]:.4f}")

In [ ]:
model.save_model(path="models/ddpm_fashion_mnist_conditional.pth")

If you do not want to train, load the pre-trained conditional model (1000 timesteps, full Fashion-MNIST training set with label dropout).

In [ ]:
model = DDPM(timesteps=TIMESTEPS, channels=CHANNELS, num_classes=NUM_CLASSES).to(device)
#model.load_model(path="models/ddpm_fashion_mnist_conditional.pth", device=device)            # for running locally
model.load_model(path="AIBIP/C4-Diffusion_Models/models/ddpm_fashion_mnist_conditional.pth", device=device)  # for running in colab
model.eval()

### Generating a Specific Class

We can now ask for any class by name. `DDPM.sample_cfg` takes a tensor of labels
and a `guidance_scale`, and returns one image per label. Here we generate one
example of **every** class — the model should produce a recognisable item for
each.

In [ ]:
model.eval()

labels = torch.arange(NUM_CLASSES, device=device)   # one of each class
samples = model.sample_cfg(labels, steps=50, guidance_scale=3.0, device=device).cpu()
samples = (samples + 1) / 2

fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(samples[i].squeeze().clamp(0, 1), cmap="gray")
    ax.set_title(FASHION_CLASSES[i], fontsize=8)
    ax.axis("off")
plt.suptitle("One sample per class (guidance scale = 3.0)", y=1.05)
plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

### The Guidance Scale

The `guidance_scale` $w$ controls how strongly samples are pushed towards the
requested class:

$$\varepsilon = \varepsilon_\text{uncond} + w \,\big(\varepsilon_\text{cond} - \varepsilon_\text{uncond}\big)$$

- $w = 0$: ignores the label entirely (**unconditional**) — samples are diverse
  but may not match the class.
- $w = 1$: plain **conditional** sampling.
- $w > 1$: **amplifies** the class signal — samples match the class more clearly,
  but diversity drops and very large $w$ can introduce artefacts.

We fix the class to a single category and sweep $w$ to see this trade-off
directly. Watch how the samples become "more typical" of the class as $w$
grows.

In [ ]:
target_class = 7   # Sneaker
n = 8
scales = [0.0, 1.0, 3.0, 5.0, 8.0]

fig, axes = plt.subplots(len(scales), n, figsize=(14, 9))
for row, w in enumerate(scales):
    labels = torch.full((n,), target_class, device=device)
    torch.manual_seed(0)  # same start noise across rows -> isolate effect of w
    samples = model.sample_cfg(labels, steps=50, guidance_scale=w, device=device).cpu()
    samples = (samples + 1) / 2
    for col in range(n):
        axes[row, col].imshow(samples[col].squeeze().clamp(0, 1), cmap="gray")
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(f"w = {w}", rotation=0, labelpad=30, fontsize=11, va="center")
plt.suptitle(f"Effect of guidance scale on class '{FASHION_CLASSES[target_class]}'", y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

### A Full Class Grid

To appreciate the control we have gained, we generate several samples for **every
class** at a moderate guidance scale. Compare this to the *unconditional* C4-2
samples, where we could not choose the category at all — here every row is a
deliberate choice.

In [ ]:
rows_per_class = 1
w = 3.0
n_cols = 8

fig, axes = plt.subplots(NUM_CLASSES, n_cols, figsize=(14, 16))
for cls in range(NUM_CLASSES):
    labels = torch.full((n_cols,), cls, device=device)
    samples = model.sample_cfg(labels, steps=50, guidance_scale=w, device=device).cpu()
    samples = (samples + 1) / 2
    for col in range(n_cols):
        axes[cls, col].imshow(samples[col].squeeze().clamp(0, 1), cmap="gray")
        axes[cls, col].axis("off")
    axes[cls, 0].set_ylabel(FASHION_CLASSES[cls], rotation=0, labelpad=45,
                            fontsize=10, va="center")
plt.suptitle("Conditional samples — all classes (guidance scale = 3.0)", y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

### From Class Labels to Text Prompts

Everything in this notebook generalises directly to **text-to-image** models:

| Fashion-MNIST here | Stable Diffusion |
|---|---|
| 10 class labels | Arbitrary text prompts |
| `nn.Embedding` for the class | A **text encoder** (CLIP) |
| Embedding added to timestep | Text embeddings via **cross-attention** |
| `guidance_scale` (CFG) | The **same** `guidance_scale` knob |
| Null class for unconditional | Empty prompt `""` for unconditional |

The conditioning mechanism and CFG are **identical** — only the *source* of the
conditioning embedding changes. The remaining ingredient for Stable Diffusion is
running diffusion in a compressed **latent space** (built by an autoencoder, as
in C2) instead of pixel space, which is what makes high-resolution generation
affordable.

---
## Try It Yourself — Experiment with Conditioning & CFG

Work in pairs. **Predict first, then run, then explain in one sentence.**

**A. Read the guidance sweep.** In the $w$-sweep grid, describe what happens to
(i) class fidelity and (ii) diversity as $w$ increases. At which $w$ do you see
the best balance for this class? Is it the same for every class?

**B. The null class.** CFG needs the model to also predict *unconditionally*. Set
`guidance_scale = 0.0` and generate with a fixed label — do the samples still
respect the class? Explain what the network is using as its label internally.

**C. Over-guiding.** Push `guidance_scale` to 15 or 20. The samples should look
over-saturated or distorted. Why does amplifying the conditional signal too much
hurt image quality? (Hint: $w$ extrapolates *beyond* the conditional prediction.)

**D. Label dropout matters.** The model was trained with `p_uncond=0.1`. Predict
what would happen to CFG if we had trained with `p_uncond=0.0` (never showing the
null class). Why would `sample_cfg` then fail to work properly?